# MVEN hyperparameter tuning: Mumbai PM10 (30- and 90-day horizons)

This notebook tunes the **Multivariate Engression Network (MVEN)** of Pathak & Chakraborty (2026) for probabilistic forecasting of daily PM10 concentrations at 17 air-quality monitoring stations in Mumbai, India. It covers the complete workflow, from downloading the data to saving the final trained models, for two forecast horizons: **30 days** (short term) and **90 days** (long term).

**Workflow**

| Section | Content |
|---|---|
| 1 | Environment and configuration |
| 2 | Data: download, checks, spatial information |
| 3 | Chronological split and preprocessing |
| 4 | Model and hyperparameter search space |
| 5 | Validation objective (CRPS) |
| 6 | Hyperparameter optimisation with Optuna |
| 7 | Tuning results |
| 8 | Final models: retraining and saving |
| 9 | Output files |

**Protocol (both horizons)**

| Item | Setting |
|---|---|
| Data | daily PM10 (µg/m³), 17 stations, 1 Jan 2021 – 31 Dec 2024, used as published |
| Test set | last H days (H = 30 or 90); not used anywhere in this notebook |
| Validation set | the V days before the test set (V = 60 for H = 30, V = 120 for H = 90) |
| Input window | 60 days, all stations |
| Training loss | energy score (β = 1, M = 2 samples per window), Adam, batch size 64 |
| Search | Optuna, Tree-structured Parzen Estimator, 100 trials × 100 epochs per horizon |
| Objective | CRPS of 100-sample forecasts on the validation set |
| Final models | best configuration retrained on training + validation data and saved |

**Running the notebook.** It runs on Kaggle or locally (Python ≥ 3.10). A GPU is recommended: the full search takes roughly 1.5–3 hours per horizon on an NVIDIA T4. Set `QUICK_RUN = True` in Section 1.2 to check the workflow in a few minutes (the results are then not meaningful). All outputs are written to the folder `MVEN_tuning/`.

**References.** Pathak, R. & Chakraborty, T. (2026). Deep generative spatiotemporal engression for probabilistic forecasting of epidemics. *Transactions on Machine Learning Research*. · Shen, X. & Meinshausen, N. (2025). Engression: extrapolation through the lens of distributional regression. *JRSS-B*, 87(3), 653–677. · Akiba, T. et al. (2019). Optuna: a next-generation hyperparameter optimization framework. *KDD*, 2623–2631. · Data: [GraphSVR repository](https://github.com/ctanujit/GraphSVR/tree/main/Data).

---
## 1. Environment and configuration
### 1.1 Packages

In [ ]:
import subprocess, sys

# Install the packages that are not available yet
for package in ["stengression", "optuna"]:
    try:
        __import__(package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import os, json, time, random, hashlib, shutil, warnings
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
import optuna
from stengression import MVEN, energy_score_loss, SpatioTemporalDataset

warnings.filterwarnings("ignore", message=".*dropout option adds dropout after all but last recurrent layer.*")
optuna.logging.set_verbosity(optuna.logging.WARNING)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
for p in ["stengression", "torch", "optuna", "numpy", "pandas"]:
    print(f"{p:>12}: {version(p)}")

### 1.2 Configuration
All settings are collected here. The values follow the protocol above and should not be changed for a reproduction of the study.

In [ ]:
MODEL_NAME = "MVEN"

# Data (GraphSVR repository)
DATA_URL     = "https://raw.githubusercontent.com/ctanujit/GraphSVR/main/Data/Mumbai_PM10_Data.csv"
STATIONS_URL = "https://raw.githubusercontent.com/ctanujit/GraphSVR/main/Data/Mumbai_Stations_PM10.csv"
START_DATE   = "2021-01-01"

# Forecasting design
HORIZONS    = [30, 90]            # forecast horizons in days
VAL_LENGTHS = {30: 60, 90: 120}   # validation length for each horizon
LOOKBACK    = 60                  # input window in days

# Training
EPOCHS        = 100
BATCH_SIZE    = 64
TRAIN_SAMPLES = 2                 # M, forecast samples per window in the energy score
EVAL_SAMPLES  = 100               # forecast samples used to compute CRPS

# Hyperparameter search
N_TRIALS          = 100
TUNING_SEED       = 0             # fixed seed for every trial and for the sampler
FINAL_SEED        = 1             # seed of the saved final models
TIME_BUDGET_HOURS = 11.0          # stop tuning before a 12-hour session limit; a re-run resumes

QUICK_RUN = False                 # True: 3 trials x 2 epochs, only to check the workflow
if QUICK_RUN:
    N_TRIALS, EPOCHS = 3, 2
    print("QUICK_RUN is on: the results are not meaningful.")

# Output folders
OUTPUT_DIR = Path("/kaggle/working" if os.path.isdir("/kaggle/working") else ".") / f"{MODEL_NAME}_tuning"
MODEL_DIR = OUTPUT_DIR / "models"
FIGURE_DIR = OUTPUT_DIR / "figures"
for folder in [OUTPUT_DIR, MODEL_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print("Outputs:", OUTPUT_DIR.resolve())

### 1.3 Reproducibility
`set_seed` fixes the random number generators of Python, NumPy and PyTorch. The `stengression` model classes also set their own fixed initialisation seeds internally; we therefore pass seed-dependent values to them when building a model (Section 4), so that the seed controls the initial weights, the training noise and the forecast samples.

In [ ]:
def set_seed(seed):
    # Fix all random number generators used in this notebook
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

---
## 2. Data
### 2.1 Download

In [ ]:
raw = pd.read_csv(DATA_URL)
stations = pd.read_csv(STATIONS_URL).iloc[:, :3].dropna().reset_index(drop=True)
stations.columns = ["station", "latitude", "longitude"]

# The data file has one column per station (same order as the station list) and no dates
data = raw.copy()
data.columns = stations["station"]
data.index = pd.date_range(START_DATE, periods=len(data), freq="D", name="date")

# Keep a copy of the data and a fingerprint, so later notebooks can check they use the same file
raw.to_csv(OUTPUT_DIR / "Mumbai_PM10_Data.csv", index=False)
stations.to_csv(OUTPUT_DIR / "Mumbai_Stations_PM10.csv", index=False)
DATA_MD5 = hashlib.md5(raw.to_csv(index=False).encode()).hexdigest()

station_names = list(data.columns)
n_days, n_stations = data.shape
print(f"{n_days} days ({data.index[0].date()} to {data.index[-1].date()}), {n_stations} stations, MD5 {DATA_MD5[:12]}")
stations

### 2.2 Data check
Summary statistics per station, missing values, and the share of values that lie on straight-line segments of at least five days. Such segments are interpolations made by the data provider; they are kept as published.

In [ ]:
def interpolated_share(series, min_length=5):
    # Share of days on straight-line segments of at least `min_length` days
    flat = (series.diff().diff().abs() < 1e-4).astype(int)
    runs = flat.groupby((flat != flat.shift()).cumsum()).transform("sum") * flat
    return float((runs >= min_length - 2).mean())

check = pd.DataFrame({
    "mean": data.mean(), "sd": data.std(), "min": data.min(), "max": data.max(),
    "missing": data.isna().sum(), "interpolated (%)": [100 * interpolated_share(data[s]) for s in station_names],
}).round(1)
print(f"Missing values: {int(data.isna().sum().sum())}, negative values: {int((data < 0).sum().sum())}, "
      f"mean correlation between stations: {data.corr().values[np.triu_indices(n_stations, 1)].mean():.2f}")
check

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.8))
ax.plot(data.index, data.values, color="grey", lw=0.4, alpha=0.5)
ax.plot(data.index, data.mean(axis=1), color="black", lw=1, label="mean of 17 stations")
ax.axhline(100, color="tab:red", ls="--", lw=1, label="India 24-hour PM10 standard (100 µg/m³)")
ax.set_ylabel("PM10 (µg/m³)")
ax.set_title("Daily PM10 at 17 Mumbai monitoring stations")
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "pm10_series.png", dpi=200)
plt.show()

### 2.3 Spatial information
Pairwise great-circle (Haversine) distances between the stations, in kilometres.
MVEN does not use them: it forecasts each station from its own history with one shared network. They are shown here for completeness.

In [ ]:
lat = np.radians(stations["latitude"].values)
lon = np.radians(stations["longitude"].values)
a = (np.sin((lat[:, None] - lat[None, :]) / 2) ** 2
     + np.cos(lat[:, None]) * np.cos(lat[None, :]) * np.sin((lon[:, None] - lon[None, :]) / 2) ** 2)
distance_km = 2 * 6371 * np.arcsin(np.sqrt(a))

pairs = distance_km[np.triu_indices(n_stations, 1)]
print(f"Distances between stations: {pairs.min():.1f} to {pairs.max():.1f} km (median {np.median(pairs):.1f} km)")

fig, ax = plt.subplots(figsize=(5.5, 6.5))
ax.scatter(stations["longitude"], stations["latitude"], c=data.mean(), cmap="YlOrRd", s=80, edgecolor="black")
for _, row in stations.iterrows():
    ax.annotate(row["station"].split(",")[0], (row["longitude"], row["latitude"]), fontsize=7, xytext=(4, 2),
                textcoords="offset points")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title("Monitoring stations (colour = mean PM10)")
ax.set_aspect(1 / np.cos(np.radians(19.1)))
fig.tight_layout()
fig.savefig(FIGURE_DIR / "station_map.png", dpi=200)
plt.show()

---
## 3. Chronological split and preprocessing
For each horizon the series is split in time order. The **test set** (last H days) is removed before anything else and is not used in this notebook. Each station is standardised with the mean and standard deviation of the data the model is trained on, and the training data are cut into windows of 60 input days and H target days. Validation forecasts start at non-overlapping origins aligned to the end of the validation set.

In [ ]:
values = torch.tensor(data.values, dtype=torch.float32).unsqueeze(-1)       # (days, stations, 1)

def prepare_horizon(horizon):
    test_start = n_days - horizon
    val_start = test_start - VAL_LENGTHS[horizon]
    train = values[:val_start]
    mean, std = train.mean(0), train.std(0)
    scaled = (values[:test_start] - mean) / std                              # training + validation, scaled
    return {
        "horizon": horizon, "val_start": val_start, "test_start": test_start,
        "mean": mean, "std": std, "scaled": scaled,
        "train_loader": DataLoader(SpatioTemporalDataset(scaled[:val_start], LOOKBACK, horizon),
                                   batch_size=BATCH_SIZE, shuffle=False),
        "val_origins": sorted(range(test_start - horizon, val_start - 1, -horizon)),
    }

prepared = {h: prepare_horizon(h) for h in HORIZONS}

split_table = pd.DataFrame([{
    "horizon (days)": h,
    "training": f"{data.index[0].date()} to {data.index[p['val_start'] - 1].date()}",
    "validation": f"{data.index[p['val_start']].date()} to {data.index[p['test_start'] - 1].date()}",
    "test (not used)": f"{data.index[p['test_start']].date()} to {data.index[-1].date()}",
    "training windows": len(p["train_loader"].dataset),
    "validation origins": ", ".join(str(data.index[o].date()) for o in p["val_origins"]),
} for h, p in prepared.items()])
split_table.to_csv(OUTPUT_DIR / "splits.csv", index=False)
split_table

---
## 4. Model and hyperparameter search space
**MVEN** adds random noise to the standardised 60-day input and passes it through an LSTM shared by all stations; a linear layer outputs the H-day forecast. Each station is forecast from its own history (no spatial module). Each new noise draw gives a different possible future, so repeated draws form the forecast distribution (pre-additive engression).

| Hyperparameter | Search space |
|---|---|
| LSTM hidden size | 2 – 100 |
| LSTM layers | 1 – 5 |
| dropout between LSTM layers | 0.0 – 0.5 |
| learning rate | 10⁻⁵ – 10⁻¹ (log scale) |
| noise distribution | Gaussian, uniform |

In [ ]:
def build_model(params, horizon, seed):
    model = MVEN(
        in_feat_dim=1, num_nodes=n_stations,
        lstm_hidden_dim=params["hidden"], lstm_num_layers=params["layers"],
        lstm_dropout=params["dropout"] if params["layers"] > 1 else 0.0,
        p_lag=LOOKBACK, t_pred=horizon, noise_dist=params["noise_dist"],
        temporal_seed=9 + seed,                                   # seed-dependent LSTM initialisation
    )
    return model.to(device), None                                 # MVEN uses no spatial weights

def suggest_params(trial):
    return {
        "hidden": trial.suggest_int("hidden", 2, 100),
        "layers": trial.suggest_int("layers", 1, 5),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "lr": trial.suggest_float("lr", 1e-5, 1e-1, log=True),
        "noise_dist": trial.suggest_categorical("noise_dist", ["gaussian", "uniform"]),
    }

def train_model(model, W_list, loader, lr, epochs):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.fit(loader, optimizer, energy_score_loss, num_epochs=epochs, m_samples=TRAIN_SAMPLES,
              device=device, monitor=False, visualize=False)

def forward(model, x, W_list):
    return model(x)

---
## 5. Validation objective
For each validation origin, the trained model forecasts the next H days 100 times (100 noise draws, computed in one batched forward pass). The forecasts are transformed back to µg/m³ and scored with the continuous ranked probability score,

$$\mathrm{CRPS} = \mathbb{E}|X - y| - \tfrac{1}{2}\,\mathbb{E}|X - X'|,$$

averaged over days, stations and origins (lower is better).

In [ ]:
@torch.no_grad()
def forecast_samples(model, W_list, history, prep, n_samples=EVAL_SAMPLES):
    # history: (LOOKBACK, stations, 1), scaled. Returns (n_samples, horizon, stations) in µg/m³
    model.eval()
    x = history.unsqueeze(0).repeat(n_samples, 1, 1, 1).to(device)
    out = forward(model, x, W_list).cpu()
    return (out * prep["std"] + prep["mean"])[..., 0]

def crps(samples, observed):
    term1 = (samples - observed).abs().mean(0)
    term2 = (samples.unsqueeze(0) - samples.unsqueeze(1)).abs().mean((0, 1))
    return (term1 - 0.5 * term2).mean().item()

def validation_crps(model, W_list, prep):
    scores = []
    for origin in prep["val_origins"]:
        samples = forecast_samples(model, W_list, prep["scaled"][origin - LOOKBACK:origin], prep)
        scores.append(crps(samples, values[origin:origin + prep["horizon"], :, 0]))
    return float(np.mean(scores))

def make_objective(prep):
    def objective(trial):
        params = suggest_params(trial)
        set_seed(TUNING_SEED)
        model, W_list = build_model(params, prep["horizon"], TUNING_SEED)
        train_model(model, W_list, prep["train_loader"], params["lr"], EPOCHS)
        score = validation_crps(model, W_list, prep)
        return score if np.isfinite(score) else 1e10      # a diverged run gets a very poor score
    return objective

---
## 6. Hyperparameter optimisation
One Optuna study per horizon (100 trials, Tree-structured Parzen Estimator sampler). The studies are stored in `optuna.db`; if a session ends before all trials are finished, running the notebook again with the previous output available continues the remaining trials. The time budget is shared equally between the horizons still to be tuned.

In [ ]:
DB_PATH = OUTPUT_DIR / "optuna.db"

# On Kaggle: continue from this notebook's previous output, if it was attached as an input
previous = [p for p in Path("/kaggle/input").glob(f"**/{MODEL_NAME}_tuning/optuna.db")] if Path("/kaggle/input").exists() else []
if previous and not DB_PATH.exists():
    shutil.copy(previous[0], DB_PATH)
    print("Continuing from", previous[0])

storage = f"sqlite:///{DB_PATH}"
studies, tuning_minutes = {}, {}
start = time.time()
for i, h in enumerate(HORIZONS):
    name = f"{MODEL_NAME}_PM10_H{h}" + ("_quick" if QUICK_RUN else "")
    study = optuna.create_study(study_name=name, storage=storage, direction="minimize", load_if_exists=True,
                                sampler=optuna.samplers.TPESampler(seed=TUNING_SEED))
    done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    remaining_time = TIME_BUDGET_HOURS * 3600 - (time.time() - start)
    timeout = max(60, remaining_time / (len(HORIZONS) - i))
    print(f"H = {h}: {done} trials done, running up to {N_TRIALS - done} more (time limit {timeout / 3600:.1f} h)")
    t0 = time.time()
    if done < N_TRIALS:
        study.optimize(make_objective(prepared[h]), n_trials=N_TRIALS - done, timeout=timeout, show_progress_bar=True)
    tuning_minutes[h] = (time.time() - t0) / 60
    studies[h] = study
    print(f"H = {h}: best validation CRPS {study.best_value:.3f} after "
          f"{sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)} trials\n")

---
## 7. Tuning results
### 7.1 Selected hyperparameters
The best configuration of each horizon is saved in `best_hyperparameters_MVEN.json`, together with the validation score, the data fingerprint and the split. This is the file used by the testing notebook.

In [ ]:
best = {"model": MODEL_NAME, "dataset": "Mumbai PM10", "data_md5": DATA_MD5, "stations": station_names,
        "lookback": LOOKBACK, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "train_samples": TRAIN_SAMPLES,
        "eval_samples": EVAL_SAMPLES, "n_trials": N_TRIALS, "tuning_seed": TUNING_SEED, "quick_run": QUICK_RUN,
        "versions": {p: version(p) for p in ["stengression", "torch", "optuna"]},
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "horizons": {}}

for h, study in studies.items():
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    p = prepared[h]
    best["horizons"][str(h)] = {
        "params": study.best_params, "best_val_crps": study.best_value, "best_trial": study.best_trial.number,
        "trials_complete": len(completed), "complete": len(completed) >= N_TRIALS,
        "tuning_minutes": round(tuning_minutes[h], 1),
        "validation_period": [str(data.index[p["val_start"]].date()), str(data.index[p["test_start"] - 1].date())],
        "test_period": [str(data.index[p["test_start"]].date()), str(data.index[-1].date())],
    }
    trials = study.trials_dataframe(attrs=("number", "value", "params", "state", "duration"))
    trials.columns = [c.replace("params_", "") for c in trials.columns]
    trials.to_csv(OUTPUT_DIR / f"trials_{MODEL_NAME}_H{h}.csv", index=False)

with open(OUTPUT_DIR / f"best_hyperparameters_{MODEL_NAME}.json", "w") as f:
    json.dump(best, f, indent=2)

pd.DataFrame({f"H = {h}": {**r["params"], "validation CRPS": round(r["best_val_crps"], 3),
                           "trials": r["trials_complete"]} for h, r in best["horizons"].items()})

### 7.2 Optimisation history and hyperparameter importance
Left: validation CRPS of every trial and the best value so far. Right: the relative importance of each hyperparameter for the validation CRPS (fANOVA, as implemented in Optuna).

In [ ]:
for h, study in studies.items():
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    scores = np.array([t.value for t in completed])
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    axes[0].scatter([t.number for t in completed], scores, s=12, color="tab:blue", alpha=0.6, label="trial")
    axes[0].plot([t.number for t in completed], np.minimum.accumulate(scores), color="tab:red", label="best so far")
    if len(scores) > 3:
        axes[0].set_ylim(scores.min() * 0.95, np.percentile(scores, 90) * 1.05)
    axes[0].set_xlabel("trial"); axes[0].set_ylabel("validation CRPS (µg/m³)")
    axes[0].set_title(f"{MODEL_NAME}, H = {h}: optimisation history"); axes[0].legend(fontsize=8)
    if len(completed) >= 5:
        try:
            importance = optuna.importance.get_param_importances(study)
            axes[1].barh(list(importance)[::-1], list(importance.values())[::-1], color="tab:green")
            axes[1].set_xlabel("relative importance")
        except Exception as error:
            axes[1].text(0.5, 0.5, f"importance not available:\n{error}", ha="center", va="center", fontsize=8)
    axes[1].set_title(f"{MODEL_NAME}, H = {h}: hyperparameter importance")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"tuning_H{h}.png", dpi=200)
    plt.show()

---
## 8. Final models
### 8.1 Retraining on training + validation data
For each horizon, the selected configuration is trained from scratch on the training and validation data together (seed `FINAL_SEED`), with the same number of epochs as during tuning. The model weights, the scaling statistics and the configuration are saved, so the model can be reloaded without retraining.

In [ ]:
final_models = {}
for h in HORIZONS:
    params = studies[h].best_params
    p = prepared[h]
    trainval = values[:p["test_start"]]
    mean, std = trainval.mean(0), trainval.std(0)                       # statistics of training + validation
    scaled = (trainval - mean) / std
    loader = DataLoader(SpatioTemporalDataset(scaled, LOOKBACK, h), batch_size=BATCH_SIZE, shuffle=False)

    set_seed(FINAL_SEED)
    model, W_list = build_model(params, h, FINAL_SEED)
    t0 = time.time()
    train_model(model, W_list, loader, params["lr"], EPOCHS)
    train_seconds = time.time() - t0

    torch.save(model.state_dict(), MODEL_DIR / f"{MODEL_NAME}_H{h}.pth")
    config = {"model": MODEL_NAME, "horizon": h, "lookback": LOOKBACK, "params": params, "seed": FINAL_SEED,
              "trained_on": [str(data.index[0].date()), str(data.index[p["test_start"] - 1].date())],
              "scaling_mean": mean[:, 0].tolist(), "scaling_std": std[:, 0].tolist(),
              "stations": station_names, "data_md5": DATA_MD5, "train_seconds": round(train_seconds, 1)}
    with open(MODEL_DIR / f"{MODEL_NAME}_H{h}_config.json", "w") as f:
        json.dump(config, f, indent=2)
    final_models[h] = {"scaled": scaled, "mean": mean, "std": std}
    print(f"H = {h}: trained in {train_seconds / 60:.1f} min, saved {MODEL_NAME}_H{h}.pth and its configuration")

### 8.2 Loading a saved model
The saved files are enough to rebuild the model and forecast. As a check, each model is reloaded and produces a 100-sample forecast starting after the last training day (the period that the testing notebook evaluates). Only the forecast is shown here; the observed test values are deliberately not used in this notebook.

In [ ]:
def load_model(horizon):
    with open(MODEL_DIR / f"{MODEL_NAME}_H{horizon}_config.json") as f:
        config = json.load(f)
    model, W_list = build_model(config["params"], horizon, config["seed"])
    model.load_state_dict(torch.load(MODEL_DIR / f"{MODEL_NAME}_H{horizon}.pth", map_location=device))
    return model, W_list, config

show = list(data.mean().sort_values().index[[0, 5, 11, 16]])          # four stations from low to high PM10
fig, axes = plt.subplots(len(HORIZONS), 4, figsize=(16, 3.2 * len(HORIZONS)), squeeze=False)
for i, h in enumerate(HORIZONS):
    model, W_list, config = load_model(h)
    f = final_models[h]
    set_seed(FINAL_SEED)
    samples = forecast_samples(model, W_list, f["scaled"][-LOOKBACK:], {"mean": f["mean"], "std": f["std"]}).numpy()
    low, median, high = np.quantile(samples, [0.025, 0.5, 0.975], axis=0)
    history_days = data.index[n_days - h - LOOKBACK:n_days - h]
    forecast_days = data.index[n_days - h:]
    for j, st in enumerate(show):
        k = station_names.index(st)
        ax = axes[i, j]
        ax.plot(history_days, data[st].iloc[n_days - h - LOOKBACK:n_days - h], color="black", lw=1, label="last 60 training days")
        ax.fill_between(forecast_days, low[:, k], high[:, k], color="tab:green", alpha=0.25, label="95% prediction interval")
        ax.plot(forecast_days, median[:, k], color="tab:green", lw=1.3, label="median forecast")
        ax.set_title(f"{st}, H = {h}", fontsize=9)
        ax.tick_params(axis="x", labelrotation=30, labelsize=7)
        if j == 0:
            ax.set_ylabel("PM10 (µg/m³)")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(FIGURE_DIR / "final_model_forecasts.png", dpi=200)
plt.show()

---
## 9. Output files
| File | Content |
|---|---|
| `best_hyperparameters_MVEN.json` | selected hyperparameters and validation CRPS for each horizon (input of the testing notebook) |
| `trials_MVEN_H30.csv`, `trials_MVEN_H90.csv` | every Optuna trial with its hyperparameters and validation CRPS |
| `optuna.db` | the Optuna studies (allows resuming) |
| `splits.csv` | dates of the training, validation and test sets |
| `models/MVEN_H30.pth`, `models/MVEN_H90.pth` | weights of the final models |
| `models/MVEN_H*_config.json` | configuration, seed and scaling statistics needed to reload a model |
| `figures/` | series, station map, tuning history and importance, final-model forecasts |
| `Mumbai_PM10_Data.csv`, `Mumbai_Stations_PM10.csv` | the data used |

In [ ]:
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(OUTPUT_DIR)):<45} {path.stat().st_size / 1024:8.1f} kB")
print("\nArchive:", archive)